In [ ]:
%reload_ext autoreload
%autoreload 2
import os
from pathlib import Path
import sys
from typing import Any, Literal, Optional

from qecbench import TaskStats

# Add parent directory to the path
sys.path.insert(0, os.path.abspath('..'))

# Register learned decoders so they can be instantiated by `qecdec.decoders.create_decoder`.
import learned_decoders  # noqa: F401
from utils import get_csv_path

In [4]:
import matplotlib.pyplot as plt

import plot_utils

%matplotlib inline
plt.rcParams.update({"figure.figsize": (10, 6), "font.size": 12, "pdf.fonttype": 42})

In [6]:
LIST_NUM_RELAYS = [1, 2, 4, 8, 16]
LIST_NUM_CHAINS = [1, 2, 4, 8, 16]

# Helper functions

In [5]:
def load_stats(
    *,
    circuit_name: str,
    circuit_params: dict[str, Any],
    error_rates: list[float],
    decoder_name: str,
    decoder_fixed_params: dict[str, Any],
    decoder_flex_params: dict[str, list[Any]],
) -> list[TaskStats]:
    filtered_stats: list[TaskStats] = []
    csv_path = get_csv_path(circuit_name, circuit_params, decoder_name)
    all_stats = TaskStats.load_csv(csv_path)
    for s in all_stats:
        m = s.metadata
        if m.error_rate not in error_rates:
            continue
        if not decoder_fixed_params.items() <= m.decoder_params.items():
            continue
        if not all(m.decoder_params[k] in v_list for k, v_list in decoder_flex_params.items()):
            continue
        filtered_stats.append(s)
    return filtered_stats


In [ ]:
def plot_fr_heatmap(
    *,
    circuit_name: str,
    circuit_params: dict[str, Any],
    multirelaybp_fixed_params: dict[str, Any],
    learned_multirelaybp_fixed_params: dict[str, Any],
    error_rates: list[float],
    all_num_relays: list[int],
    all_num_chains: list[int],
    fr_mode: Literal["per_shot", "per_round"],
    suptitle: Optional[str] = None,
    save_at: Optional[str | Path] = None,
):
    # upper row: MultiRelayBP
    # lower row: LearnedMultiRelayBP
    # column: one for each error rate
    nrows, ncols = 2, len(error_rates)
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(5 * ncols + 1, 5 * nrows), squeeze=False
    )
    cmap = plt.cm.viridis.copy()
    cmap.set_bad("white")

    first_row_stats = load_stats(
        circuit_name=circuit_name,
        circuit_params=circuit_params,
        error_rates=error_rates,
        decoder_name="MultiRelayBP",
        decoder_fixed_params=multirelaybp_fixed_params,
        decoder_flex_params={
            "num_relays": all_num_relays,
            "num_chains": all_num_chains,
        },
    )
    second_row_stats = load_stats(
        circuit_name=circuit_name,
        circuit_params=circuit_params,
        error_rates=error_rates,
        decoder_name="LearnedMultiRelayBP",
        decoder_fixed_params=learned_multirelaybp_fixed_params,
        decoder_flex_params={
            "num_relays": all_num_relays,
            "num_chains": all_num_chains,
        },
    )

    if suptitle:
        fig.suptitle(suptitle, fontsize=plot_utils.SUPTITLE_FONTSIZE)
    fig.tight_layout()
    if save_at:
        fig.savefig(save_at)
    plt.show()